In [ ]:
!pip install llmlingua torch transformers openai tiktoken

In [2]:
import os
import time
import tiktoken
from openai import OpenAI
from llmlingua import PromptCompressor

In [3]:
# ==========================================
# 1. SETUP CONFIGURATION & CLIENTS
# ==========================================
from google.colab import userdata

# Configure your custom OpenAI-compatible endpoint
API_KEY = userdata.get('GROQ_API_KEY')        # Replace with your API key
BASE_URL = "https://api.groq.com/openai/v1"   # Replace with your custom endpoint URL (e.g., vLLM, Ollama)
MODEL_NAME = "openai/gpt-oss-20b"             # Replace with your model name on the server

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# Token counter helper using tiktoken (cl100k_base or gpt-4 tokenizer)
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))

In [5]:
# ==========================================
# 2. INITIALIZE LLMLINGUA-2 COMPRESSOR
# ==========================================

print("Loading LLMLingua-2 model...")
compressor = PromptCompressor(
    model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
    use_llmlingua2=True,
    device_map="cpu"  # Forces PyTorch to load and run the BERT model on CPU
)
print("Compressor successfully loaded!")


# ==========================================
# 3. HELPER FUNCTIONS WITH TIMING
# ==========================================

def call_llm(prompt: str) -> tuple[str, float]:
    """Calls the LLM API at temperature 0 and measures generation latency."""
    start_time = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    end_time = time.perf_counter()
    gen_time_ms = (end_time - start_time) * 1000
    output_text = response.choices[0].message.content

    return output_text, gen_time_ms


def compress_prompt(prompt: str, target_rate: float = 0.5) -> tuple[str, float]:
    """Compresses prompt using LLMLingua-2 and measures compression latency."""
    start_time = time.perf_counter()

    results = compressor.compress_prompt_llmlingua2(
        prompt,
        rate=target_rate,
        force_tokens=['\n', '.', '!', '?', ','],
        drop_consecutive=True
    )

    end_time = time.perf_counter()
    comp_time_ms = (end_time - start_time) * 1000

    return results['compressed_prompt'], comp_time_ms

Loading LLMLingua-2 model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Compressor successfully loaded!


In [6]:
# ==========================================
# 4. DEFINE PROMPT & QUESTION
# ==========================================

context = """
John: So, um, I've been thinking about the project, you know, and I believe we need to, uh, make some changes.
I mean, we want the project to succeed, right? So, like, I think we should consider maybe revising the timeline.
We have been facing several bottlenecks in the frontend development pipeline due to API integration delays.
Sarah: I totally agree, John. I mean, we have to be realistic, you know. The timeline is, like, too tight.
You know what I mean? We should definitely extend it by at least two weeks to allow for proper QA testing.
Alex: That makes sense. However, if we shift the QA period, we must make sure the deployment phase isn't pushed
past Q3. Otherwise, the stakeholders will raise concerns about our quarterly commitments.
"""

question = "What main proposal did John and Sarah agree on, and what was Alex's condition?"

full_prompt = f"Context:\n{context}\n\nQuestion: {question}"

In [9]:
# ==========================================
# 5. RUN EXPERIMENT & BENCHMARK
# ==========================================

print("\n=== RUNNING BASELINE (UNCOMPRESSED) ===")
orig_token_count = count_tokens(full_prompt)
baseline_output, baseline_gen_time = call_llm(full_prompt)

print(f"Original Prompt Tokens : {orig_token_count}")
print(f"Generation Time        : {baseline_gen_time:.2f} ms")


print("\n=== RUNNING COMPRESSOR & COMPRESSED CALL ===")
# Compress prompt to target 50% token reduction (rate=0.5)
compressed_prompt, compression_time = compress_prompt(full_prompt, target_rate=0.7)
compressed_token_count = count_tokens(compressed_prompt)

# Send compressed prompt to LLM
comp_output, compressed_gen_time = call_llm(compressed_prompt)

print(f"Compressed Prompt Tokens : {compressed_token_count} (Saved {orig_token_count - compressed_token_count} tokens)")
print(f"Compression Overhead Time: {compression_time:.2f} ms")
print(f"Generation Time          : {compressed_gen_time:.2f} ms")
print(f"Total Time (Comp + Gen)  : {compression_time + compressed_gen_time:.2f} ms")


=== RUNNING BASELINE (UNCOMPRESSED) ===
Original Prompt Tokens : 194
Generation Time        : 455.36 ms

=== RUNNING COMPRESSOR & COMPRESSED CALL ===
Compressed Prompt Tokens : 132 (Saved 62 tokens)
Compression Overhead Time: 7418.17 ms
Generation Time          : 435.01 ms
Total Time (Comp + Gen)  : 7853.18 ms


In [10]:
# ==========================================
# 6. RESULTS ANALYSIS
# ==========================================

print("\n" + "="*50)
print("               SUMMARY COMPARISON             ")
print("="*50)
print(f"{'Metric':<25} | {'Baseline':<15} | {'Compressed':<15}")
print("-" * 60)
print(f"{'Input Token Count':<25} | {orig_token_count:<15} | {compressed_token_count:<15}")
print(f"{'Compression Overhead':<25} | {'N/A':<15} | {f'{compression_time:.2f} ms':<15}")
print(f"{'LLM Generation Time':<25} | {f'{baseline_gen_time:.2f} ms':<15} | {f'{compressed_gen_time:.2f} ms':<15}")
print(f"{'Total End-to-End Time':<25} | {f'{baseline_gen_time:.2f} ms':<15} | {f'{(compression_time + compressed_gen_time):.2f} ms':<15}")
print("-" * 60)

print("\n--- BASELINE OUTPUT ---")
print(baseline_output)

print("\n--- COMPRESSED OUTPUT ---")
print(comp_output)

print("\n--- COMPRESSED PROMPT TEXT PASSED TO MODEL ---")
print(compressed_prompt)


               SUMMARY COMPARISON             
Metric                    | Baseline        | Compressed     
------------------------------------------------------------
Input Token Count         | 194             | 132            
Compression Overhead      | N/A             | 7418.17 ms     
LLM Generation Time       | 455.36 ms       | 435.01 ms      
Total End-to-End Time     | 455.36 ms       | 7853.18 ms     
------------------------------------------------------------

--- BASELINE OUTPUT ---
John and Sarah agreed to **extend the project timeline by at least two weeks** to give the QA testing enough time.  
Alex’s condition was that, even with the extended QA period, the **deployment phase must still finish before the end of Q3** so that quarterly commitments are met.

--- COMPRESSED OUTPUT ---
John and Sarah agreed to **extend the project timeline by at least two weeks** to allow for proper QA testing.  
Alex’s condition is that **the deployment phase must still finish within Q